# Enterprise RAG — Hands-On, Part 1 of 11: The corpus and its permissions

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

---
# Part 1 - The corpus and its permissions

Content and permissions are two separate feeds, joined by `doc_id`:

- **Content** - `data/corpus/*.md`. Frontmatter carries only `doc_id` and `title`; the body is the
  document text. No access-control field lives here.
- **Permissions** - `data/acl_manifest.json`. One JSON record per `doc_id`: `sensitivity`,
  `allowed_groups`, `region`, `source`, `need_to_know`, `contains_pii`, `valid_from`/`valid_until`.
  This is the stand-in for whatever system actually owns entitlements in production - an admin
  console, an HR/entitlements system, a Confluence-space-permissions export.

`load_corpus()` joins the two by `doc_id`. A content file with no matching manifest record is
refused outright - there is no "default to internal" fallback. Getting that join wrong is the number
one cause of enterprise RAG leaks, which is why it lives in its own, boringly explicit function.

**Where this ends up:** the join happens once, at ingest time (Part 4). From there the resulting
`ResourceAttributes` are written to *two* places with different jobs - a denormalised copy on each
chunk in the vector index (a cache, used only to make retrieval cheap) and a row in a separate local
ACL catalog (SQLite), which is the *authoritative* copy the post-retrieval policy check actually
reads.

In [ ]:
from enterprise_rag.ingest.loader import load_corpus

docs = load_corpus()
print(f"{len(docs)} documents\n")
print(f"{'doc_id':<16}{'source':<12}{'sensitivity':<14}{'region':<8}{'allowed_groups'}")
print("-" * 92)
for d in sorted(docs, key=lambda x: (x.attrs.source, x.attrs.doc_id)):
    a = d.attrs
    extra = ""
    if a.need_to_know:
        extra += f"  need-to-know={a.need_to_know}"
    if a.valid_from:
        extra += f"  embargoed until {a.valid_from}"
    if a.contains_pii:
        extra += "  [PII]"
    print(f"{a.doc_id:<16}{a.source:<12}{a.sensitivity:<14}{a.region:<8}"
          f"{','.join(a.allowed_groups)}{extra}")

In [ ]:
# The manifest is the source of truth for access control; frontmatter no longer carries it.
print((SETTINGS.corpus_dir / "PM-2026-03-14.md").read_text(encoding="utf-8"))

One document's content frontmatter, next to its permissions record - two files, one `doc_id`.

In [ ]:
import json

manifest = json.loads(SETTINGS.acl_manifest_file.read_text(encoding="utf-8"))
record = next(r for r in manifest["documents"] if r["doc_id"] == "PM-2026-03-14")
print(json.dumps(record, indent=2))

### The people

Eight personas, each chosen to exercise a *different* policy rule. Note the last one: a principal from
another tenant holding **every** group and the highest clearance. It is the negative control - it must
never see anything at all.

In [ ]:
from enterprise_rag.identity import list_principals

header = (
    f"{'user_id':<22} {'role':<24} {'clearance':<13} {'region':<6} "
    f"{'pii':^3} {'ext':^3}  groups"
)
print(header)
print("-" * len(header))

for p in list_principals():
    print(
        f"{p.user_id:<22} {p.role:<24} {p.clearance:<13} {p.region:<6} "
        f"{'Y' if p.can_view_pii else '.':^3} {'Y' if p.is_external else '.':^3}  "
        f"{', '.join(p.groups)}"
    )
    if p.projects:
        print(f"{'':<22} {'':<24} {'':<13} {'':<6} {'':^3} {'':^3}  "
              f"projects: {', '.join(p.projects)}")

print("\npii/ext: Y = yes, . = no")

---

**Next ▶:** [2. The policy engine](part02-policy-engine.ipynb)
